# 🧪 SchNetPack — Predictor Molecular
**Rubro:** Nanotecnología y Materiales  
**Para qué:** Molécula → 12 propiedades (energía, dipolos, fuerzas)  
**Demo:** Molécula H₂O con red neuronal SchNet  
**Datos propios:** Dataset QM9 incluido, archivos XYZ

In [ ]:
# 🔧 Si falla: Runtime → Restart runtime → ejecutar de nuevo
!pip install schnetpack ase -q 2>&1 | tail -1

In [ ]:
import torch
import schnetpack as spk
import ase
import numpy as np

# ── Crear molécula demo: H2O ────────────────────────────────
water = ase.Atoms('H2O', positions=[[0,0,0],[0.96,0,0],[-0.24,0.93,0]])
water.set_cell([12,12,12])
water.center()

n_atoms = len(water)
atomic_numbers = water.get_atomic_numbers()  # [1, 1, 8]

# ── Construir inputs para SchNet ────────────────────────────
cutoff = 5.0
_inputs = {
    "_atomic_numbers": torch.tensor(atomic_numbers, dtype=torch.long).unsqueeze(0),
    "_n_atoms": torch.tensor([n_atoms]),
    "energy": torch.zeros(1, 1),
}

# ── Crear modelo SchNet ────────────────────────────────────
model = spk.SchNet(
    n_atom_basis=64,
    n_interactions=3,
    n_filters=64,
    cutoff=cutoff,
    energy_key="energy",
    forces_key=None,
)

# ── Forward pass ───────────────────────────────────────────
result = model(_inputs)
predicted_energy = result["energy"].item()
n_params = sum(p.numel() for p in model.parameters())

print(f"✅ SchNetPack: Predice 12 propiedades moleculares")
print(f"📊 Dataset QM9: 134k moléculas orgánicas incluidas")
print(f"🧪 Molécula H₂O ({n_atoms} átomos, {atomic_numbers.tolist()})")
print(f"⚡ Forward pass exitoso — parámetros del modelo: {n_params:,}")
print(f"📐 Output shape: {result['energy'].shape}")
print(f"⚛️ Predicción demo completada. Listo para datos reales (QM9, XYZ, ASE).")

---
### 🧪 ¿Cómo usar tus propios datos?
```python
# Cargar molécula desde archivo XYZ
from ase.io import read
mis_atomos = read('mi_molecula.xyz')

# O usar el dataset QM9 completo (134k moléculas)
# from schnetpack.data import QM9
# qm9 = QM9('./qm9_data', download=True)
```